In [ ]:
import weaviate
from weaviate.classes.config import Configure, Property, DataType
import pickle , json
import numpy as np
import os
from tqdm import tqdm
from src.sentencetransformers.st_class import SentenceTransformersEmbeddings
from dotenv import load_dotenv

load_dotenv('../.env.example/.env')

True

In [3]:
import weaviate
from weaviate.classes.config import Configure, Property, DataType
import pickle , json
import numpy as np
import os
from tqdm import tqdm
from src.sentencetransformers.st_class import SentenceTransformersEmbeddings
from dotenv import load_dotenv

load_dotenv('.env.example/.env')

True

In [4]:
embedding_model = SentenceTransformersEmbeddings('sentence-transformers/all-mpnet-base-v2')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 710.87it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
# Best practice: store your credentials in environment variables
weaviate_url = os.environ["WEAVIATE_URL"]
weaviate_api_key = os.environ["WEAVIATE_API_KEY"]

In [ ]:
# Step 1.1: Connect to your Weaviate Cloud instance
with weaviate.connect_to_weaviate_cloud(
    cluster_url=weaviate_url,
    auth_credentials=weaviate_api_key,
) as client:
    
    # Step 1.2: Create a collection
    movies = client.collections.create(
        name="Movie",
        vector_config=Configure.Vectors.self_provided(),  # No automatic vectorization since we're providing vectors
    )



    # Step 1.3: Import three objects
    data_objects = [
        {"properties": {"title": "The Matrix", "description": "A computer hacker learns about the true nature of reality and his role in the war against its controllers.", "genre": "Science Fiction"},
        "vector": [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]},
        {"properties": {"title": "Spirited Away", "description": "A young girl becomes trapped in a mysterious world of spirits and must find a way to save her parents and return home.", "genre": "Animation"},
        "vector": [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]},
        {"properties": {"title": "The Lord of the Rings: The Fellowship of the Ring", "description": "A meek Hobbit and his companions set out on a perilous journey to destroy a powerful ring and save Middle-earth.", "genre": "Fantasy"},
        "vector": [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]}
    ]

    # Insert the objects with vectors
    movies = client.collections.get("Movie")
    with movies.batch.fixed_size(batch_size=200) as batch:
        for obj in data_objects:
            batch.add_object(properties=obj["properties"], vector=obj["vector"])

    print(
        f"Imported {len(data_objects)} objects with vectors into the Movie collection"
    )

In [3]:

# Step 2.1: Connect to your Weaviate Cloud instance
with weaviate.connect_to_weaviate_cloud(
    cluster_url=weaviate_url,
    auth_credentials=weaviate_api_key,
) as client:

    # Step 2.2: Use this collection
    movies = client.collections.use("Movie")

    # Step 2.3: Perform a vector search with NearVector
    response = movies.query.near_vector(
        near_vector=[0.11, 0.21, 0.31, 0.41, 0.51, 0.61, 0.71, 0.81], 
        limit=2
    )

    for obj in response.objects:
        print(json.dumps(obj.properties, indent=2))  # Inspect the results

{
  "genre": "Science Fiction",
  "description": "A computer hacker learns about the true nature of reality and his role in the war against its controllers.",
  "title": "The Matrix"
}
{
  "genre": "Animation",
  "title": "Spirited Away",
  "description": "A young girl becomes trapped in a mysterious world of spirits and must find a way to save her parents and return home."
}


In [8]:
with open('../dataset/metadata_chunks.pkl','rb') as m:
    metadata_chunks = pickle.load(m)

In [8]:
metadata_chunks[:2]

[{'celex': '32019D0276',
  'act_name': 'Decision (EU) 2019/276 of the European Parliament and of the Council of 12 December 2018 on the mobilisation of the Flexibility Instrument to reinforce key programmes for the competitiveness of the EU and to finance immediate budgetary measures to address the ongoing challenges of migration, refugee inflows and security threats',
  'act_type': 'Decision',
  'eurovoc': 'aid to refugees; budget appropriation; EC general budget; European security; refugee; migratory flow; Community financing; Community migration policy; Community expenditure; commitment appropriation',
  'subject_matter': 'cooperation policy;  budget;  EU finance;  international security;  migration',
  'legal_basis': '32013Q1220(01)',
  'date': '2018-12-12',
  'authors': 'European Parliament; European Council',
  'status': 'In Force',
  'cites': '32013R1311',
  'treaty': 'TFEU',
  'additional_info': 'No additional information',
  'chunk_number': 1,
  'total_chunks': 2,
  'document_

In [ ]:
metordata= metadata_chunks[50000]
for key , value in metordata.items():
    print(f"{key}: {value}")

celex: 32011D0890
act_name: 2011/890/EU: Commission Implementing Decision of 22Ã December 2011 providing the rules for the establishment, the management and the functioning of the network of national responsible authorities on eHealth
act_type: Decision_IMPL
eurovoc: Euronet; automatic information system; health; exchange of information; digital public service
subject_matter: communications;  information and information processing;  health;  executive power and public service
legal_basis: 32011L0024
date: 2011-12-22
authors: European Commission
status: In Force
cites: 32006D1639; 32002L58; 32001D844; 32007D1350; 31995L46
treaty: TFEU (2008)
additional_info: No additional information
chunk_number: 1
total_chunks: 4
document_length: 8293


In [7]:
with open('../dataset/text_chunks.pkl','rb') as c:
    text_chunks = pickle.load(c)

In [ ]:
for index , chunk in enumerate(text_chunks[:20]):
    print(f"index : {index}")
    print(chunk)

index : 0
22.2.2019 EN Official Journal of the European Union L 54/3 DECISION (EU) 2019/276 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL of 12 December 2018 on the mobilisation of the Flexibility Instrument to reinforce key programmes for the competitiveness of the EU and to finance immediate budgetary measures to address the ongoing challenges of migration, refugee inflows and security threats THE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE EUROPEAN UNION, Having regard to the Treaty on the Functioning of the European Union, Having regard to the Interinstitutional Agreement of 2 December 2013 between the European Parliament, the Council and the Commission on budgetary discipline, on cooperation in budgetary matters and on sound financial management (1), and in particular point 12 thereof, Having regard to the proposal from the European Commission, Whereas, (1) The Flexibility Instrument is intended to allow the financing of clearly identified expenditure which could not be financed wi

In [6]:
with open('../dataset/embeddings.npy','rb') as e:
    emmbedings = np.load(e)    

In [ ]:
type(emmbedings)

numpy.ndarray

In [1]:
print("live")

live


In [7]:
print(f" this is len meta: {len(metadata_chunks)}")
print(f" this is len chunks: {len(text_chunks)}")
print(f" this is len embeddings: {len(emmbedings)}")

 this is len meta: 694515
 this is len chunks: 694515
 this is len embeddings: 694515


# Cloud

In [6]:
weaviate_client = weaviate.connect_to_weaviate_cloud(
    cluster_url=weaviate_url,
    auth_credentials=weaviate_api_key,
)

In [7]:
weaviate_client.is_connected()

True

In [4]:
eur_laws = weaviate_client.collections.create(
    name="Euro_Laws",
    vector_config=Configure.Vectors.self_provided(),    
    properties=[
        Property(name="text", data_type=DataType.TEXT),
        Property(name="celex", data_type=DataType.TEXT),
        Property(name="act_name", data_type=DataType.TEXT),
        Property(name="act_type", data_type=DataType.TEXT),
        Property(name="eurovoc", data_type=DataType.TEXT),
        Property(name="subject_matter", data_type=DataType.TEXT),
        Property(name="legal_basis", data_type=DataType.TEXT),
        Property(name="authors", data_type=DataType.TEXT),
        Property(name="status", data_type=DataType.TEXT),
        Property(name="cites", data_type=DataType.TEXT),
        Property(name="treaty", data_type=DataType.TEXT),
        Property(name="additional_info", data_type=DataType.TEXT),
        Property(name="chunk_number", data_type=DataType.INT),
        Property(name="total_chunks", data_type=DataType.INT),
        Property(name="document_length", data_type=DataType.INT),
    ],
)

In [10]:
# Batch insert
eur_laws = weaviate_client.collections.get("Euro_Laws")

with eur_laws.batch.fixed_size(batch_size=128) as batch:
    for i in tqdm(range(len(text_chunks))):
        text = text_chunks[i]
        meta = metadata_chunks[i]
        vector = emmbedings[i].tolist()

        properties = {
            "text": text,
            "celex": str(meta.get("celex", "")),
            "act_name": str(meta.get("act_name", "")),
            "act_type": str(meta.get("act_type", "")), 
            "eurovoc": str(meta.get("eurovoc", "")),
            "subject_matter": str(meta.get("subject_matter", "")),
            "legal_basis": str(meta.get("legal_basis", "")),
            "authors": str(meta.get("authors", "")),
            "status": str(meta.get("status", "")),
            "cites": str(meta.get("cites", "")),
            "treaty": str(meta.get("treaty", "")),
            "additional_info": str(meta.get("additional_info", "")),
            "chunk_number": int(meta.get("chunk_number", 0)),
            "total_chunks": int(meta.get("total_chunks", 0)),
            "document_length": int(meta.get("document_length", 0)),
        }

        batch.add_object(
            properties=properties,
            vector=vector
        )

  0%|          | 425/694515 [00:00<02:48, 4129.22it/s]

100%|██████████| 694515/694515 [49:19<00:00, 234.70it/s]  


In [4]:
with weaviate.connect_to_weaviate_cloud(
    cluster_url=weaviate_url,
    auth_credentials=weaviate_api_key,
) as client:

    # Step 2.2: Use this collection
    Euro_Laws = client.collections.use("Euro_Laws")

    # Step 2.3: Perform a vector search with NearVector
    response = Euro_Laws.query.near_vector(
        near_vector= vector , 
        limit=10
    )

    for obj in response.objects:
        print(json.dumps(obj.properties, indent=10))  # Inspect the results

{
          "cites": "32011L36; teu_2016/art_5; char_2016; dec_framw/2002/946; tfeu_2016/art_325; 32011L93; 32014L62; 32017L1371; 32009L123; teu_2016/pro_21; tfeu_2016/pro_22; 32013L40; teu_2016/pro_22; dec_framw/2008/841; teu_2016/art_2; 32015L849; tfeu_2016/pro_21; 31997F0625%2801%29; 32014L42; 32002D187; 32008L99; dec_framw/2001/500; dec_framw/2004/757; 32014L57; dec_framw/2009/948; dec_framw/2001/413; dec_framw/2003/568; 32017L541",
          "subject_matter": "politics and public safety;  European construction;  criminal law;  free movement of capital;  social affairs;  international security",
          "act_type": "Directive",
          "chunk_number": 5,
          "eurovoc": "elimination of terrorism; EU police and customs cooperation; penalty; financial transaction; international criminal law; crime prevention; European security; international crime; European Judicial Network in criminal matters; laundering of funds",
          "act_name": "Directive (EU) 2018/1673 of the Euro

In [8]:
vector = embedding_model.embed_query("Drug dealing Sentences")

In [10]:
Euro_Laws = weaviate_client.collections.use("Euro_Laws")

# Step 2.3: Perform a vector search with NearVector
response = Euro_Laws.query.near_vector(
    near_vector= vector , 
    limit=10
)

for obj in response.objects:
    print(json.dumps(obj.properties, indent=10)) 

{
          "celex": "32004F0757",
          "total_chunks": 6,
          "status": "In Force",
          "text": "when the offender has supplied the competent authorities with valuable information. (7) It is necessary to take measures to enable the confiscation of the proceeds of the offences referred to in this Framework Decision. (8) Measures should be taken to ensure that legal persons can be held liable for the criminal offences referred to by this Framework Decision which are committed for their benefit. (9) The effectiveness of the efforts made to tackle illicit drug trafficking depends essentially on the harmonisation of the national measures implementing this Framework Decision, HAS DECIDED AS FOLLOWS: Article 1 Definitions For the purposes of this Framework Decision: 1. drugs: shall mean any of the substances covered by the following United Nations Conventions: (a) the 1961 Single Convention on Narcotic Drugs (as amended by the 1972 Protocol); (b) the 1971 Vienna Convention o

# locallll

In [ ]:
!docker-compose up -d

In [8]:
import weaviate
from weaviate.classes.config import Configure

# Step 1.1: Connect to your local Weaviate instance
with weaviate.connect_to_local() as client:

    # Step 1.2: Create a collection
    movies = client.collections.create(
        name="Movie",
        vector_config=Configure.Vectors.self_provided(),  # No automatic vectorization since we're providing vectors
    )

    # Step 1.3: Import three objects
    data_objects = [
        {"properties": {"title": "The Matrix", "description": "A computer hacker learns about the true nature of reality and his role in the war against its controllers.", "genre": "Science Fiction"},
        "vector": [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]},
        {"properties": {"title": "Spirited Away", "description": "A young girl becomes trapped in a mysterious world of spirits and must find a way to save her parents and return home.", "genre": "Animation"},
        "vector": [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]},
        {"properties": {"title": "The Lord of the Rings: The Fellowship of the Ring", "description": "A meek Hobbit and his companions set out on a perilous journey to destroy a powerful ring and save Middle-earth.", "genre": "Fantasy"},
        "vector": [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]}
    ]

    # Insert the objects with vectors
    movies = client.collections.get("Movie")
    with movies.batch.fixed_size(batch_size=200) as batch:
        for obj in data_objects:
            batch.add_object(properties=obj["properties"], vector=obj["vector"])

    print(
        f"Imported {len(data_objects)} objects with vectors into the Movie collection"
    )

Imported 3 objects with vectors into the Movie collection


In [9]:
import weaviate
import json

# Step 2.1: Connect to your local Weaviate instance
with weaviate.connect_to_local() as client:

    # Step 2.2: Use this collection
    movies = client.collections.use("Movie")

    # Step 2.3: Perform a vector search with NearVector
    response = movies.query.near_vector(
        near_vector=[0.11, 0.21, 0.31, 0.41, 0.51, 0.61, 0.71, 0.81], 
        limit=2
    )

    for obj in response.objects:
        print(json.dumps(obj.properties, indent=2))  # Inspect the results

{
  "title": "The Matrix",
  "description": "A computer hacker learns about the true nature of reality and his role in the war against its controllers.",
  "genre": "Science Fiction"
}
{
  "title": "Spirited Away",
  "description": "A young girl becomes trapped in a mysterious world of spirits and must find a way to save her parents and return home.",
  "genre": "Animation"
}


# OUR RAG

In [7]:
import weaviate
from weaviate.classes.config import Configure, Property, DataType
import pickle , json
import numpy as np
import os
from tqdm import tqdm
from dotenv import load_dotenv

load_dotenv()

True

In [14]:
with weaviate.connect_to_local() as client:

    # Delete collection if it exists
    if client.collections.exists("Euro_Laws"):
        print("Deleting existing Euro_Laws collection...")
        client.collections.delete("Euro_Laws")

Deleting existing Euro_Laws collection...


In [ ]:
with weaviate.connect_to_local() as client:

    # Step 1.2: Create a collection
    eur_laws = client.collections.create(
        name="Euro_Laws",
        vector_config=Configure.Vectors.self_provided(),    
        properties=[
            Property(name="text", data_type=DataType.TEXT),
            Property(name="celex", data_type=DataType.TEXT),
            Property(name="act_name", data_type=DataType.TEXT),
            Property(name="act_type", data_type=DataType.TEXT),
            Property(name="eurovoc", data_type=DataType.TEXT),
            Property(name="subject_matter", data_type=DataType.TEXT),
            Property(name="legal_basis", data_type=DataType.TEXT),
            Property(name="authors", data_type=DataType.TEXT),
            Property(name="status", data_type=DataType.TEXT),
            Property(name="cites", data_type=DataType.TEXT),
            Property(name="treaty", data_type=DataType.TEXT),
            Property(name="additional_info", data_type=DataType.TEXT),
            Property(name="chunk_number", data_type=DataType.INT),
            Property(name="total_chunks", data_type=DataType.INT),
            Property(name="document_length", data_type=DataType.INT),
        ],
    )



In [13]:
with weaviate.connect_to_local() as client:        
    # Batch insert
    eur_laws = client.collections.get("Euro_Laws")

    with eur_laws.batch.fixed_size(batch_size=64) as batch:
        for i in tqdm(range(len(text_chunks))):
            text = text_chunks[i]
            meta = metadata_chunks[i]
            vector = emmbedings[i].tolist()

            properties = {
                "text": text,
                "celex": str(meta.get("celex", "")),
                "act_name": str(meta.get("act_name", "")),
                "act_type": str(meta.get("act_type", "")), 
                "eurovoc": str(meta.get("eurovoc", "")),
                "subject_matter": str(meta.get("subject_matter", "")),
                "legal_basis": str(meta.get("legal_basis", "")),
                "authors": str(meta.get("authors", "")),
                "status": str(meta.get("status", "")),
                "cites": str(meta.get("cites", "")),
                "treaty": str(meta.get("treaty", "")),
                "additional_info": str(meta.get("additional_info", "")),
                "chunk_number": int(meta.get("chunk_number", 0)),
                "total_chunks": int(meta.get("total_chunks", 0)),
                "document_length": int(meta.get("document_length", 0)),
            }

            batch.add_object(
                properties=properties,
                vector=vector
            )

 31%|███       | 213401/694515 [05:28<20:25, 392.67it/s]  {'message': 'Failed to send 1 in a batch of 64', 'errors': {'store is read-only due to: disk usage too high. Set to read-only at 90.04%, threshold set to 90.00%'}}
{'message': 'Failed to send 1 objects in a batch of 64. Please inspect client.batch.failed_objects or collection.batch.failed_objects for the failed objects.'}
{'message': 'Failed to send 1 in a batch of 64', 'errors': {'store is read-only due to: disk usage too high. Set to read-only at 90.04%, threshold set to 90.00%'}}
{'message': 'Failed to send 1 objects in a batch of 64. Please inspect client.batch.failed_objects or collection.batch.failed_objects for the failed objects.'}
 31%|███       | 213504/694515 [05:28<22:59, 348.74it/s]{'message': 'Failed to send 1 in a batch of 64', 'errors': {'store is read-only due to: disk usage too high. Set to read-only at 90.04%, threshold set to 90.00%'}}
{'message': 'Failed to send 1 objects in a batch of 64. Please inspect cli

# SentenceTransformer

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
embedding_model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

query_embedding = embedding_model.encode([enhanced_query])


In [13]:
# Step 2.1: Connect to your local Weaviate instance
with weaviate.connect_to_local() as client:

    # Step 2.2: Use this collection
    movies = client.collections.use("Euro_Laws")

    # Step 2.3: Perform a vector search with NearVector
    response = movies.query.near_vector(
        near_vector= vector , 
        limit=10
    )

    for obj in response.objects:
        print(json.dumps(obj.properties, indent=10))  # Inspect the results

{
          "subject_matter": "politics and public safety;  European construction;  criminal law;  free movement of capital;  social affairs;  international security",
          "cites": "32011L36; teu_2016/art_5; char_2016; dec_framw/2002/946; tfeu_2016/art_325; 32011L93; 32014L62; 32017L1371; 32009L123; teu_2016/pro_21; tfeu_2016/pro_22; 32013L40; teu_2016/pro_22; dec_framw/2008/841; teu_2016/art_2; 32015L849; tfeu_2016/pro_21; 31997F0625%2801%29; 32014L42; 32002D187; 32008L99; dec_framw/2001/500; dec_framw/2004/757; 32014L57; dec_framw/2009/948; dec_framw/2001/413; dec_framw/2003/568; 32017L541",
          "treaty": "TFEU",
          "status": "In Force",
          "authors": "European Council; European Parliament",
          "celex": "32018L1673",
          "eurovoc": "elimination of terrorism; EU police and customs cooperation; penalty; financial transaction; international criminal law; crime prevention; European security; international crime; European Judicial Network in criminal

In [3]:
!docker run -d \
  -p 8080:8080 \
  -p 50051:50051 \
  -v weaviate_data:/var/lib/weaviate \
  semitechnologies/weaviate:latest


/usr/local/python/3.12.1/lib/python3.12/pty.py:95: DeprecationWarning: This process (pid=15779) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


8a81c560719f47a62be5dbd59dcadbe8f2f9d33492ed9b434339a4607888037b
